In [ ]:


import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE, RF_PARAM_5G

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('../config/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

selected_campaigns = list(range(1, 20))
# Data filtering
df = filter_dataframe(
    df=df,
    include_columns=['pci', 'beam_index', 'nr_arfcn', 'operator_id', 'rsrq', 'sinr', 'rssi', 'rsrp'],
    campaigns=selected_campaigns,
)

In [ ]:

from scripts.matrix_operations import create_point_matrix
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scripts.utils import extract_unique_npcis
import pandas as pd
from concurrent.futures import ThreadPoolExecutor


def train_kmeans(df: pd.DataFrame, n_clusters: int, random_state: int, rf_param, unique_npcis, feature_mode: str):
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    if feature_mode == 'position':
        df_features = df[['lat', 'lng']].values
    elif feature_mode == 'radio':
        df_features, _ = create_point_matrix(df, unique_npcis, rf_param)
    else:
        print(f'Invalid feature mode: {feature_mode}')
        return

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    cluster_labels = kmeans.fit_predict(df_features)

    df['cluster_labels'] = cluster_labels

    # Calculate inertia
    inertia = kmeans.inertia_

    # Calculate silhouette score
    silhouette_avg = silhouette_score(df_features, cluster_labels)

    # Calculate Davies-Bouldin score
    davies_bouldin = davies_bouldin_score(df_features, cluster_labels)

    return inertia, silhouette_avg, davies_bouldin,


print('KMEANS PERFORMANCE')

# Example usage
unique_npcis = extract_unique_npcis(df['measurements_matrix'])
rf_param = RF_PARAM_5G.RSRQ

data = []
feature_modes = ['position', 'radio']
max_clusters = 5
n_runs = 2
for mode in feature_modes:
    for n_clusters in range(2, max_clusters + 1):
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = [
                executor.submit(
                    train_kmeans,
                    df,
                    n_clusters,
                    42 * i,
                    rf_param,
                    unique_npcis,
                    mode,
                )
                for i in range(n_runs)
            ]
            for f in futures:
                r = f.result()
                data.append((n_clusters, mode) + r)
                print(f'Testing with k={n_clusters} mode {mode}')

# Define columns
cols = ['K', 'Feature', 'Inertia', 'Silhouette Score', 'Davies-Bouldin Score']

# Create DataFrame
res_df = pd.DataFrame(data, columns=cols)

In [ ]:
post_mask = res_df['Feature'] == 'position'

print("Clustering with LAT LNG positions")
print(res_df[post_mask].groupby(['Feature', 'K']).mean().round(3))

print("Clustering with RF value positions")
print(res_df[~post_mask].groupby(['Feature', 'K']).mean().round(3))